# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [24]:
# Write your code below.
import os
from dotenv import load_dotenv

load_dotenv()

PRICE_DATA = os.getenv("PRICE_DATA")



In [25]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)




For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [27]:
# Write your code below.
import dask.dataframe as dd


dd_prices = dd.read_parquet(parquet_files)

dd_prices = dd_prices.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x["Close"].shift(1),
        Adj_Close_lag_1=x["Adj Close"].shift(1),
        Returns=(x["Close"] / x["Close"].shift(1)) - 1,
        hi_lo_range=x["High"] - x["Low"]
    )
)
dd_prices



C:\Users\mghaf\AppData\Local\Temp\ipykernel_12228\331486282.py:7: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_prices = dd_prices.groupby("Ticker", group_keys=False).apply(


,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
npartitions=13078,,,,,,,,,,,,
,"datetime64[ns, UTC]",float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [28]:
# Write your code below.
df_stock_features = dd_prices.compute()


# Adding in 10-day moving average of returns
df_stock_features["ten_day_avg_returns"] = df_stock_features.groupby("Ticker")["Returns"].transform(lambda x: x.rolling(10).mean())


In [29]:
df_stock_features

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ten_day_avg_returns
Ticker,,,,,,,,,,,,,
DOV,2003-01-02 00:00:00+00:00,13.402376,20.418425,20.773701,19.567097,19.627426,766183.0,2003,NaN,NaN,NaN,1.206604,NaN
DOV,2003-01-03 00:00:00+00:00,13.362774,20.358093,20.472050,20.170399,20.398314,624612.0,2003,20.418425,13.402376,-0.002955,0.301651,NaN
DOV,2003-01-06 00:00:00+00:00,13.732376,20.921175,21.068649,20.344687,20.344687,958028.0,2003,20.358093,13.362774,0.027659,0.723963,NaN
DOV,2003-01-07 00:00:00+00:00,13.587173,20.699965,20.947989,20.559195,20.921175,776626.0,2003,20.921175,13.732376,-0.010574,0.388794,NaN
DOV,2003-01-08 00:00:00+00:00,13.265972,20.210619,20.699965,20.177103,20.699965,659073.0,2003,20.699965,13.587173,-0.023640,0.522861,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
CTLT,2012-12-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2012,NaN,NaN,NaN,NaN,NaN
CTLT,2012-12-26 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2012,NaN,NaN,NaN,NaN,NaN
CTLT,2012-12-27 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2012,NaN,NaN,NaN,NaN,NaN


In [30]:
df_stock_features["ten_day_avg_returns"].shape

(3173427,)

In [31]:
df_stock_features["ten_day_avg_returns"].isna().sum()

392435

In [32]:
df_stock_features["ten_day_avg_returns"].notna().sum()

2780992

In [33]:
df_stock_features["ten_day_avg_returns"].describe()

count    2.780992e+06
mean     5.243778e-03
std      3.056782e-01
min     -1.807723e-01
25%     -2.721821e-03
50%      7.006422e-04
75%      4.011826e-03
max      1.450594e+02
Name: ten_day_avg_returns, dtype: float64

In [34]:
df_stock_features.describe()


Price,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ten_day_avg_returns
count,2.792542e+06,2.792542e+06,2.792542e+06,2.792542e+06,2.792542e+06,2.792542e+06,3.173427e+06,2.792085e+06,2.792085e+06,2.791387e+06,2.792542e+06,2.780992e+06
mean,7.121454e+01,8.066665e+01,8.158663e+01,7.970649e+01,8.065681e+01,8.049651e+06,2.012044e+03,8.066183e+01,7.120947e+01,5.228012e-03,1.880143e+00,5.243778e-03
std,1.834912e+02,1.862905e+02,1.884262e+02,1.841289e+02,1.862596e+02,4.712249e+07,7.233401e+00,1.862530e+02,1.834527e+02,9.649242e-01,5.028896e+00,3.056782e-01
min,3.052100e-02,3.052100e-02,3.052100e-02,2.697900e-02,3.020800e-02,0.000000e+00,2.000000e+03,3.052100e-02,3.052100e-02,-9.982798e-01,0.000000e+00,-1.807723e-01
25%,1.659500e+01,2.362000e+01,2.394000e+01,2.328500e+01,2.361500e+01,9.398000e+05,2.006000e+03,2.362000e+01,1.659480e+01,-9.114872e-03,4.747086e-01,-2.721821e-03
50%,3.367482e+01,4.427000e+01,4.478000e+01,4.374000e+01,4.426000e+01,2.144200e+06,2.012000e+03,4.427000e+01,3.367364e+01,5.010630e-04,9.000015e-01,7.006422e-04
75%,7.182652e+01,8.343000e+01,8.431000e+01,8.250000e+01,8.343000e+01,5.092900e+06,2.018000e+03,8.343000e+01,7.182162e+01,1.016513e-02,1.814232e+00,4.011826e-03
max,9.924400e+03,9.924400e+03,9.964770e+03,9.794000e+03,9.914170e+03,9.230856e+09,2.025000e+03,9.924400e+03,9.924400e+03,1.450505e+03,6.572100e+02,1.450594e+02


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

No, it wasn't necessary.
+ Would it have been better to do it in Dask? Why?

In this specific case, given the dataset size,  I believe Dask would have been more efficient and faster.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.